In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

feats = pd.read_csv("data/processed/features.csv")
FULL = [c for c in feats.columns if c not in ("id", "label", "source", "split")]
REDUCED = ["hu1", "rect_fill", "circle_fill", "n_vertices_4",
           "rect_aspect", "area", "hu3", "solidity", "hu2"]
MINIMAL = ["n_vertices_4", "circle_fill", "rect_fill", "rect_aspect", "hu1"]

tr = feats[feats["split"] == "train"]
ytr = tr["label"]

models = {
    "LogReg": make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, class_weight="balanced")),
    "DecisionTree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=0),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "SVM": make_pipeline(StandardScaler(), SVC(C=10, kernel="rbf", class_weight="balanced")),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0),
}

cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for fs_name, fs in [("full", FULL), ("reduced", REDUCED), ("minimal", MINIMAL)]:
    for name, m in models.items():
        r = cross_validate(m, tr[fs], ytr, cv=cv, scoring=["accuracy", "f1_macro"])
        rows.append({"features": fs_name, "model": name, "n_features": len(fs),
                     "accuracy": r["test_accuracy"].mean(),
                     "f1_macro": r["test_f1_macro"].mean(),
                     "f1_std": r["test_f1_macro"].std()})

res = pd.DataFrame(rows)
print(res.pivot(index="model", columns="features", values="f1_macro").round(4))
res.to_csv("outputs/cv_results.csv", index=False)

features        full  minimal  reduced
model                                 
DecisionTree  1.0000      1.0      1.0
KNN           0.9994      1.0      1.0
LogReg        1.0000      1.0      1.0
RandomForest  1.0000      1.0      1.0
SVM           1.0000      1.0      1.0


In [3]:
from sklearn.model_selection import cross_val_score
import hashlib

# shuffled labels should score near chance (about 0.2)
rng = np.random.default_rng(0)
print("shuffled labels:", cross_val_score(models["RandomForest"], tr[MINIMAL],
      rng.permutation(ytr.values), cv=cv, scoring="f1_macro").mean().round(3))

# how good is each single feature?
for f in MINIMAL:
    s = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=0),
                        tr[[f]], ytr, cv=cv, scoring="f1_macro").mean()
    print(f, round(s, 3))

# identical silhouettes shared between train and test?
X = np.load("data/processed/silhouettes_64.npz")["X"]
h = pd.Series([hashlib.md5(x.tobytes()).hexdigest() for x in X])
print("identical train/test images:",
      len(set(h[feats.split == "train"]) & set(h[feats.split == "test"])))

shuffled labels: 0.199
n_vertices_4 0.986
circle_fill 0.999
rect_fill 0.998
rect_aspect 0.653
hu1 1.0
identical train/test images: 111


In [4]:
from sklearn.model_selection import LeaveOneGroupOut

rows = []
for fs_name, fs in [("full", FULL), ("reduced", REDUCED), ("minimal", MINIMAL)]:
    for name, m in models.items():
        r = cross_validate(m, tr[fs], ytr, groups=tr["source"], cv=LeaveOneGroupOut(),
                           scoring=["accuracy", "f1_macro"])
        rows.append({"features": fs_name, "model": name, "n_features": len(fs),
                     "accuracy": r["test_accuracy"].mean(),
                     "f1_macro": r["test_f1_macro"].mean(),
                     "f1_std": r["test_f1_macro"].std()})

res_src = pd.DataFrame(rows)
print(res_src.pivot(index="model", columns="features", values="f1_macro").round(4))
res_src.to_csv("outputs/cv_results_by_source.csv", index=False)

features        full  minimal  reduced
model                                 
DecisionTree  0.8658   0.9161   0.8661
KNN           0.8536   0.8614   0.8617
LogReg        0.9144   0.9160   0.8599
RandomForest  1.0000   1.0000   1.0000
SVM           0.8973   0.8632   0.9136


In [5]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score

SOURCES = ["generated", "kaggle_2d", "kaggle_simple"]
rows = []
for name, model in models.items():
    for src in SOURCES:
        a, b = tr[tr.source != src], tr[tr.source == src]
        p = clone(model).fit(a[MINIMAL], a.label).predict(b[MINIMAL])
        rows.append({"model": name, "held_out": src,
                     "accuracy": accuracy_score(b.label, p),
                     "f1_present": f1_score(b.label, p, labels=sorted(b.label.unique()), average="macro")})
detail = pd.DataFrame(rows)
print(detail.pivot(index="model", columns="held_out", values="accuracy").round(3))
print(detail.pivot(index="model", columns="held_out", values="f1_present").round(3))

held_out      generated  kaggle_2d  kaggle_simple
model                                            
DecisionTree      1.000        1.0          0.996
KNN               1.000        1.0          0.951
LogReg            0.999        1.0          0.997
RandomForest      1.000        1.0          1.000
SVM               1.000        1.0          0.967
held_out      generated  kaggle_2d  kaggle_simple
model                                            
DecisionTree      1.000        1.0          0.998
KNN               1.000        1.0          0.974
LogReg            0.999        1.0          0.999
RandomForest      1.000        1.0          1.000
SVM               1.000        1.0          0.982


In [6]:
rows = []
for f in FULL:
    r = {"feature": f}
    for src in SOURCES:
        a, b = tr[tr.source != src], tr[tr.source == src]
        m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(a[[f]], a.label)
        r[src] = accuracy_score(b.label, m.predict(b[[f]]))
    rows.append(r)
print(pd.DataFrame(rows).set_index("feature").round(3))

              generated  kaggle_2d  kaggle_simple
feature                                          
area              0.742      0.791          0.758
perimeter         0.544      0.596          0.288
circularity       0.922      0.834          0.943
solidity          0.587      0.736          0.561
circle_fill       0.999      0.999          0.994
rect_fill         0.999      0.995          0.976
rect_aspect       0.660      0.614          0.283
n_vertices_2      0.955      0.981          0.935
n_vertices_4      1.000      1.000          0.821
hu1               1.000      1.000          0.999
hu2               0.375      0.400          0.128
hu3               0.446      0.684          0.342
hu4               0.589      0.501          0.315
hu5               0.442      0.474          0.344
hu6               0.355      0.294          0.178
hu7               0.494      0.307          0.381
